In [1]:
!git clone https://github.com/NVlabs/CGBN

Cloning into 'CGBN'...
remote: Enumerating objects: 260, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 260 (delta 136), reused 113 (delta 112), pack-reused 70 (from 1)
Receiving objects: 100% (260/260), 148.92 KiB | 12.41 MiB/s, done.
Resolving deltas: 100% (162/162), done.


In [2]:
!git clone -b dev-dist-spectrum https://github.com/UCLA-Communications-Systems-Lab/elf-tbcc-spectrum.git

Cloning into 'elf-tbcc-spectrum'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 277 (delta 137), reused 219 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 96.39 KiB | 19.28 MiB/s, done.
Resolving deltas: 100% (137/137), done.


In [3]:
!sudo apt-get install libgmp-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libgmpxx4ldbl
Suggested packages:
  gmp-doc libgmp10-doc libmpfr-dev
The following NEW packages will be installed:
  libgmp-dev libgmpxx4ldbl
0 upgraded, 2 newly installed, 0 to remove and 51 not upgraded.
Need to get 346 kB of archives.
After this operation, 1,702 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgmpxx4ldbl amd64 2:6.2.1+dfsg-3ubuntu1 [9,580 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgmp-dev amd64 2:6.2.1+dfsg-3ubuntu1 [337 kB]
Fetched 346 kB in 0s (8,104 kB/s)   
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize fro

In [13]:
!cd elf-tbcc-spectrum/distance_spectrum_computation/ && make CONFIG=config/k31n62v6.yaml

make[1]: Entering directory '/content/elf-tbcc-spectrum/distance_spectrum_computation'
nvcc -shared -Xcompiler -fPIC -O3 -gencode arch=compute_75,code=sm_75 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_89,code=sm_89 -gencode arch=compute_89,code=compute_89 compile/distance_spectrum.cu -o lib/libfoldshift.so
make[1]: Leaving directory '/content/elf-tbcc-spectrum/distance_spectrum_computation'
K = 31 <= 64, running regular kernel with config/k31n62v6.yaml
gpu_distance_spectrum: [        1         0         0         0         0         0         0
         0         0         0       341         0      2046         0
     25482         0    250976         0   1732714         0   8585357
         0  31943020         0  90260220         0 195402486         0
 325735104         0 419804077         0 419804077         0 325735104
         0 195402486         0  90260220         0  31943020         0
   8585357         0   1732714         0    250976         0     25482
         

In [5]:
!cd elf-tbcc-spectrum/distance_spectrum_computation/ && make compile-kernel

nvcc -shared -Xcompiler -fPIC -O3 -gencode arch=compute_75,code=sm_75 -gencode arch=compute_89,code=sm_89 -gencode arch=compute_89,code=compute_89 compile/distance_spectrum.cu -o lib/libfoldshift.so


In [23]:
!cd elf-tbcc-spectrum/distance_spectrum_computation/ && make compile-cgbn

nvcc -shared -Xcompiler -fPIC -O3 -gencode arch=compute_75,code=sm_75 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_89,code=sm_89 -gencode arch=compute_89,code=compute_89 -I/content/CGBN/include -lgmp distance_spectrum_cgbn.cu -o lib/libfoldshift.so
distance_spectrum_cgbn.cu(69): warning #177-D: variable "x" was declared but never referenced
      uint32_t x = blockIdx.x * blockDim.x + threadIdx.x;
               ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

distance_spectrum_cgbn.cu(69): warning #177-D: variable "x" was declared but never referenced
      uint32_t x = blockIdx.x * blockDim.x + threadIdx.x;
               ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

distance_spectrum_cgbn.cu(69): warning #177-D: variable "x" was declared but never referenced
      uint32_t x = blockIdx.x * blockDim.x + threadIdx.x;
               ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-n

In [24]:
!cd elf-tbcc-spectrum/distance_spectrum_computation/ && python3 test_cgbn.py config/k113n254v8.yaml

Traceback (most recent call last):
  File "/content/elf-tbcc-spectrum/distance_spectrum_computation/test_cgbn.py", line 132, in <module>
    main(path)
  File "/content/elf-tbcc-spectrum/distance_spectrum_computation/test_cgbn.py", line 96, in main
    d_buf_a.copy_to_device(h_in.reshape(-1))
  File "/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devices.py", line 233, in _require_cuda_context
    return fn(*args, **kws)
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py", line 235, in copy_to_device
    _driver.host_to_device(
  File "/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/driver.py", line 3085, in host_to_device
    fn(*args)
  File "/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/driver.py", line 359, in safe_cuda_api_call
    return self._check_cuda_python_error(fname, libfn(*args))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [21]:
import numpy as np

# Load the files
ds_fold = np.load('elf-tbcc-spectrum/distance_spectrum_computation/output/fold/k31n62v6_dist_spectrum.npy')
df_cgbn = np.load('elf-tbcc-spectrum/distance_spectrum_computation/output/cgbn_out/k31n62v6_dist_spectrum.npy')

# Check for exact equality
if np.array_equal(ds_fold, df_cgbn):
    print("The files are identical.")
else:
    print("The files are different.")

The files are identical.
